<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Why Numerical Methods? Floating Point, Error & Complexity</div>
</div>

#### This is the genuinely new material for this module: the computational foundations that sit underneath every numerical and statistical technique you'll use this term. Every week from here on builds on something in this notebook.
***

## <span style="color:#B5121B">1. Computers don't do algebra</span>

A computer doesn't manipulate symbols the way you do on paper — it does arithmetic on a finite-precision approximation of a real number, many times, very fast. That distinction sounds pedantic until you see it break something you'd have sworn was correct.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

`0.1` and `0.2` cannot be represented *exactly* in binary floating point (the same way $1/3$ cannot be written exactly in decimal) — each is stored as the closest representable double-precision value, and the tiny representation errors don't cancel when you add them. This isn't a Python bug; it's IEEE 754 double precision, the standard essentially every numerical language uses.

A second, related surprise: floating-point addition isn't strictly associative.

In [ ]:
a = 1e16
b = -1e16
c = 1.0

print((a + b) + c)   # cancel the two huge numbers first, then add the small one
print(a + (b + c))   # add the small number to a huge one first

Same three numbers, same `+` operator, different grouping, different answer — one order preserves the `1.0`, the other loses it completely, because `b + c` rounds straight back to `b` (`1.0` is far too small to survive being added to $-10^{16}$ at double precision). The lesson isn't "avoid floating point" — it's "the order of operations in your code is not just a style choice, it can change your answer." Keep that in mind; it's about to matter a lot.

> **Quick check:** find a fourth number `d` such that `a + b + c + d` gives yet another different answer depending on grouping. (There's more than one way to do this.)
***

## <span style="color:#B5121B">2. Catastrophic cancellation: when a textbook-correct formula fails</span>

Suppose you're running a detector continuously and logging the arrival time of every event, in seconds, since the experiment started. After about 17 days, your timestamps are all sitting around 1.5 million seconds — but what you actually care about is the *spread* in arrival-time jitter between events, which might only be a millisecond or so.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
offset = 1.5e6          # seconds since experiment start (~17 days)
true_jitter_std = 1e-3  # true timing jitter: 1 millisecond

timestamps = offset + rng.normal(0, true_jitter_std, size=100_000)
print("first few timestamps:", timestamps[:3])
print("true variance (by construction):", true_jitter_std ** 2)

There's a well-known textbook formula for variance: $\mathrm{Var}(x) = E[x^2] - (E[x])^2$. Let's compute it that way, and compare against NumPy's built-in `np.var`:

In [ ]:
naive_var = np.mean(timestamps**2) - np.mean(timestamps)**2
numpy_var = np.var(timestamps)

print(f"naive  E[x^2]-E[x]^2 : {naive_var: .6e}")
print(f"numpy  np.var(x)     : {numpy_var: .6e}")
print(f"true value           : {true_jitter_std**2: .6e}")

**The naive formula gives a *negative* variance.** That's not a rounding quirk you can shrug off — a variance is a sum of squares, it is mathematically impossible for it to be negative. When your code hands you an impossible number, that is a very loud signal that something in the *computation* is broken, not that the underlying physics is wrong.

Here's why: `timestamps**2` is a number around $(1.5\times10^6)^2 = 2.25\times10^{12}$, and double precision carries about 15–17 significant decimal digits. The millisecond-scale jitter you actually care about is buried about nine orders of magnitude below that. `E[x^2]` and `(E[x])^2` are both huge, nearly-equal numbers — computing their difference means subtracting away all the digits that were doing useful work and being left with noise. This is **catastrophic cancellation**, and it's one of the most common ways real scientific code silently gives wrong answers.

`np.var` doesn't do this — internally it computes the mean first, then works with deviations from the mean (`x - mean(x)`), which stays at the *actual scale of the spread you care about* rather than the scale of the offset. Same statistical quantity, same underlying data, a different order of operations, and one of them is simply wrong.

### A stable, one-pass alternative: Welford's algorithm

If you can't just call `np.var` — say, because you're processing a stream of measurements one at a time and can't hold them all in memory — there's a well-known numerically stable way to compute a running variance, due to B. P. Welford (1962). It never squares the large numbers at all; it only ever works with small, well-scaled updates.

In [ ]:
def welford_variance(data):
    n = 0
    mean = 0.0
    M2 = 0.0
    for x in data:
        n += 1
        delta = x - mean
        mean += delta / n
        delta2 = x - mean
        M2 += delta * delta2
    return M2 / n

welford_var = welford_variance(timestamps)
print(f"Welford's algorithm  : {welford_var: .6e}")
print(f"true value           : {true_jitter_std**2: .6e}")

Same data, same statistical definition, a third implementation — and this one agrees with NumPy and with the true value to several significant figures. The formula on the page and the algorithm in your code are not the same thing, and this term is as much about the second one as the first.

> **Exercise:** Pick two or three different `(offset, true_jitter_std)` combinations of your own (try both much larger and much smaller offsets than $1.5\times10^6$). For each, record whether the naive formula gives a wrong-but-plausible-looking number, exactly zero, or a negative number. Does `np.var` ever get it wrong? Can you find a combination where *even Welford's algorithm* starts to struggle?
***

## <span style="color:#555656">Aside: why this module stays in Python</span>

Python trades raw execution speed for development speed — it's interpreted and dynamically typed, which is exactly why the vectorisation trick in the revision notebook gave such a large speed-up (it hands the heavy lifting to compiled C code under the hood). That trade-off is also why compiled or JIT-compiled languages — C++, Fortran, and increasingly Rust and Julia — still matter for large-scale physics simulation, where every cycle counts. This module stays in Python throughout, deliberately: it's fast enough for everything we'll do, and it's what you already know from PHYS281.
***

## <span style="color:#B5121B">3. Algorithmic complexity: how your code scales</span>

A second, related question: not "is this calculation numerically correct", but "how does its runtime grow as the dataset grows?" Suppose you have a set of star positions and want the sum of all pairwise distances between them — a calculation that shows up, in one form or another, in clustering (Week 6) and nearest-neighbour classification (Week 8).

In [ ]:
import numpy as np

def naive_pairwise_distance_sum(points):
    """Sum of all pairwise distances - naive nested-loop version."""
    n = len(points)
    total = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = points[i, 0] - points[j, 0]
                dy = points[i, 1] - points[j, 1]
                total += np.sqrt(dx**2 + dy**2)
    return total

def vectorised_pairwise_distance_sum(points):
    """Same calculation, using NumPy broadcasting instead of Python loops."""
    diff = points[:, None, :] - points[None, :, :]
    dist = np.sqrt((diff**2).sum(axis=-1))
    return dist.sum()

# sanity check: both approaches should agree
rng = np.random.default_rng(0)
test_points = rng.uniform(0, 100, size=(50, 2))
a = naive_pairwise_distance_sum(test_points)
b = vectorised_pairwise_distance_sum(test_points)
print(f"naive result      : {a:.4f}")
print(f"vectorised result : {b:.4f}")
print(f"agree to within numerical precision: {np.isclose(a, b)}")

In [ ]:
import timeit

# Single timings are noisy (OS scheduling, caching, etc.) - take the minimum
# of several repeats for each n, which is the standard way to cut through that noise.
n_values = [50, 100, 200, 400, 800]
naive_times, vec_times = [], []

for n in n_values:
    points = rng.uniform(0, 100, size=(n, 2))

    tn = min(timeit.repeat(lambda: naive_pairwise_distance_sum(points), number=1, repeat=5))
    tv = min(timeit.repeat(lambda: vectorised_pairwise_distance_sum(points), number=1, repeat=5))

    naive_times.append(tn)
    vec_times.append(tv)
    print(f"n={n:4d}   naive={tn:8.4f}s   vectorised={tv:8.5f}s   ratio={tn/tv:6.1f}x")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(n_values, naive_times, "o-", color="firebrick", label="naive nested loop")
ax.plot(n_values, vec_times, "o-", color="dodgerblue", label="vectorised (NumPy)")
ax.set_xlabel("Number of points, $n$")
ax.set_ylabel("Time (s)")
ax.set_title("Pairwise-distance calculation: runtime vs. dataset size")
ax.legend()
fig.tight_layout()
plt.show()

The naive version grows roughly as $n^2$ (double $n$, and the number of pairs to check roughly quadruples) — visible directly in how much more steeply the red curve bends upward than the blue one. The vectorised version scales far more gently over this range, because the heavy lifting happens in compiled code operating on whole arrays at once rather than in a Python-level loop. Datasets in Weeks 8 and 9 run into the thousands of rows; at that scale, the naive version isn't just slower, it can become genuinely impractical, while the vectorised version stays fast. This is the same lesson as the revision notebook's loop-vs-vectorised timing, now with a visible growth curve attached to it.

> **Exercise:** Extend the timing comparison to a third implementation using `scipy.spatial.distance.pdist`, and add it to the plot. How does it compare to the hand-written vectorised version — and can you explain, even approximately, why?
***

## <span style="color:#FD9029">Independent exercise: a real cancellation problem — nuclear binding energy</span>

Everything above used constructed examples. This one doesn't — it's real published data, and the numerical fragility is real too. Work through it in your own time; it isn't a lab-session task.

The mass of an atomic nucleus is *less* than the total mass of its separate protons and neutrons — the missing mass, converted via $E=mc^2$, is the **binding energy** that holds the nucleus together. For a nuclide with $Z$ protons and $N$ neutrons (mass number $A = Z+N$) and measured atomic mass $M$, the binding energy is

$$BE = \left[Z \cdot m(^1\mathrm{H}) + N \cdot m_n - M\right] \times 931.494\ \mathrm{MeV/u}$$

using the atomic mass of hydrogen-1 (rather than the bare proton mass) so that electron masses approximately cancel on both sides — the standard convention. `isotope_masses.csv` gives $Z$, $N$, $A$ and the measured atomic mass (in u) for eight isotopes spanning deuterium to uranium, sourced from NIST's Atomic Weights and Isotopic Compositions table.

In [ ]:
import pandas as pd
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename


m_H1 = 1.00782503223   # atomic mass of hydrogen-1, u (NIST)
m_n = 1.00866491606    # neutron mass, u (NIST CODATA)
u_to_MeV = 931.49410372  # 1 u in MeV (NIST CODATA)

isotopes = pd.read_csv(data_path("isotope_masses.csv"))
isotopes

**Part A.** Compute the binding energy per nucleon, $BE/A$, for every isotope in double precision (the default — you don't need to do anything special), and plot it against mass number $A$. If your numbers are right, you should recover a recognisable shape: this is the famous nuclear binding energy curve, and it should peak somewhere around iron/nickel — the reason fusion releases energy for light elements and fission releases energy for heavy ones.

In [ ]:
Z = isotopes["Z"].to_numpy(dtype=np.float64)
N = isotopes["N"].to_numpy(dtype=np.float64)
A = isotopes["A"].to_numpy(dtype=np.float64)
M = isotopes["atomic_mass_u"].to_numpy(dtype=np.float64)

BE = (Z * m_H1 + N * m_n - M) * u_to_MeV
BE_per_A = BE / A

for name, a_val, bea in zip(isotopes["isotope"], A, BE_per_A):
    print(f"{name:18s}  A={a_val:6.0f}   BE/A = {bea:.4f} MeV")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(A, BE_per_A, "o-", color="darkslateblue")
ax.set_xlabel("Mass number, $A$")
ax.set_ylabel("Binding energy per nucleon (MeV)")
ax.set_title("The nuclear binding energy curve, from raw NIST mass data")
fig.tight_layout()
plt.show()

**Part B.** Now redo the same calculation after casting every mass to `np.float16` (half precision, roughly 3 significant decimal digits) instead of `np.float64`. Plot both curves on the same axes. Given the sizes of the numbers involved (atomic masses up to 238, meaningful physics in the 4th–5th significant figure), what do you expect to happen *before* you run it?

In [ ]:
Z16 = isotopes["Z"].to_numpy(dtype=np.float16)
N16 = isotopes["N"].to_numpy(dtype=np.float16)
A16 = isotopes["A"].to_numpy(dtype=np.float16)
M16 = isotopes["atomic_mass_u"].to_numpy(dtype=np.float16)

BE16 = (Z16 * np.float16(m_H1) + N16 * np.float16(m_n) - M16) * np.float16(u_to_MeV)
BE_per_A16 = BE16 / A16

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(A, BE_per_A, "o-", color="darkslateblue", label="float64 (correct)")
ax.plot(A, BE_per_A16, "o--", color="firebrick", label="float16")
ax.set_xlabel("Mass number, $A$")
ax.set_ylabel("Binding energy per nucleon (MeV)")
ax.set_title("The same calculation, cast to half precision")
ax.legend()
fig.tight_layout()
plt.show()

**Part C (harder).** The float16 error doesn't grow smoothly with $A$ the way you might expect. For each isotope, compute the *fractional mass defect*, $(Z\,m(^1\mathrm{H}) + N\,m_n - M) / M$ — i.e. how small the cancellation is *relative to* the numbers being subtracted. Does the size of this fraction predict which isotopes suffer the worst float16 error? What does that tell you, in general, about when a precision choice that "should be fine" (238 easily fits in float16's range) actually isn't?

**Part D (reflection, no code).** In one or two sentences: if you inherited a real nuclear-physics codebase and found it computed binding energies in single precision (`float32`) rather than double, would you consider that a bug worth fixing? Use Part B/C's results, not just intuition, to justify your answer.

***
## Summary

Today's two lessons — that a numerically "correct" formula can still be badly wrong in practice, and that the same calculation can scale very differently depending on how it's written — aren't abstract computer-science trivia. They're the foundation the rest of this module sits on: Week 2's root-finders behave differently depending on numerical precision near a root; Week 4's error propagation depends on getting a numerical derivative right; and Week 10's discussion of how large language models work literally depends on today's floating-point lesson (the standard implementation of the softmax function subtracts off the largest value before exponentiating, for exactly the cancellation reasons covered here). Keep this notebook — you'll be pointed back to it more than once.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis &middot; Lancaster University
</div>